# Trabajo Nº1 – Clasificación de Perros y Gatos con CNN y Transfer Learning

Este notebook sirve como plantilla mínima para que puedas comenzar a construir tus propias CNN y aplicar Transfer Learning.  
Incluye:
- Configuración básica e imports.
- Carga de imágenes desde carpetas (`Cats_Dogs/cat` y `Cats_Dogs/dog`) y redimensionado a tamaño fijo.
- Celda final para generar el `.csv` de predicciones con el formato requerido.

Indicaciones:
- En la entrega se proporciona `Cats_Dogs.zip` con 1000 imágenes por clase (carpetas `cat` y `dog`).  
- Descomprime el zip en el mismo directorio del notebook antes de ejecutar.
- Tú debes definir y entrenar tu modelo (CNN propia y un modelo de Transfer Learning).  
- Al final, genera `predicciones_cats_dogs.csv` con el formato especificado.

In [36]:
# Añade aquí los import que utilices en tu código
import os
import pathlib
import shutil
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping, ModelCheckpoint
from PIL import Image
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

plt.rcParams['figure.figsize'] = (12, 6)
np.random.seed(33)
tf.random.set_seed(33)
sns.set(style="whitegrid")


## Carga de Imágenes

Se cargan exactamente **1000 imágenes de cada clase** desde la carpeta `Cats_Dogs`, asumiendo esta estructura:

```text
Cats_Dogs/
├─ cat/
│  ├─ 0.jpg
│  ├─ ...
├─ dog/
│  ├─ 0.jpg
│  ├─ ...
```

Preprocesamiento:
- Redimensionado a `224×224` píxeles (puedes cambiar `IMG_SIZE` si lo justificas).
- Conversión a RGB.

Las etiquetas se almacenan como enteros:
- `0` → gato  
- `1` → perro

In [40]:
# Configuración de rutas y parámetros
DATASET_DIR = "Cats_Dogs"  # Carpeta con subdirectorios "cat" y "dog"
CATEGORIES = ["cat", "dog"]
IMG_SIZE = (160, 160)  # Resolución elegida (suficiente para capturar rasgos sin disparar el costo computacional)
DATASET_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"

dataset_path = pathlib.Path(DATASET_DIR)
if not dataset_path.exists():
    print("No se encontró la carpeta Cats_Dogs. Se descargará un dataset de referencia...")
    zip_path = keras.utils.get_file(
        fname="cats_and_dogs_filtered.zip",
        origin=DATASET_URL,
        extract=True,
        cache_dir=".",
        cache_subdir="datasets"
    )
    extracted_dir = pathlib.Path(zip_path).with_suffix("")
    train_dir = extracted_dir / "train"
    name_mapping = {"cat": "cats", "dog": "dogs"}
    for target_name, source_name in name_mapping.items():
        src_folder = train_dir / source_name
        dst_folder = dataset_path / target_name
        dst_folder.mkdir(parents=True, exist_ok=True)
        if any(dst_folder.iterdir()):
            continue
        for image_file in src_folder.glob("*.jpg"):
            shutil.copy(image_file, dst_folder / image_file.name)
    print(f"Dataset disponible en {dataset_path.resolve()}")

X = []
y = []

# Cargar imágenes y etiquetas
for label, category in enumerate(CATEGORIES):
    folder_path = dataset_path / category
    image_files = sorted(folder_path.glob("*.jpg"))
    print(f"Leyendo {len(image_files)} imágenes de {category}...")
    for img_path in image_files:
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize(IMG_SIZE)
            X.append(np.array(img))
            y.append(label)
        except Exception as e:
            print("Error al cargar:", img_path, e)

# Conversión a arrays y normalización
X = np.array(X, dtype="float32") / 255.0
y = np.array(y, dtype="int32")

print("Forma de X:", X.shape)
print("Forma de y:", y.shape)

class_counts = {CATEGORIES[idx]: int(count) for idx, count in enumerate(np.bincount(y))}
print("Distribución de clases:", class_counts)

# Visualización rápida de ejemplos aleatorios
num_examples = min(10, len(X))
indices = np.random.choice(len(X), size=num_examples, replace=False)
fig, axes = plt.subplots(2, num_examples // 2, figsize=(14, 6))
for ax, idx in zip(axes.flatten(), indices):
    ax.imshow(X[idx])
    ax.set_title(CATEGORIES[y[idx]])
    ax.axis("off")
plt.tight_layout()
plt.show()


Forma de X: (2000, 224, 224, 3)
Forma de y: (2000,)


## Desarrollo de la Práctica

A partir de aquí se replica el flujo seguido en los apuntes del Tema 5:
1. Dividir el dataset en entrenamiento/validación/prueba.
2. Definir un bloque de *Data Augmentation* y una CNN propia con capas convolucionales, normalización y *dropout* progresivo.
3. Entrenar la red con *callbacks* (ReduceLROnPlateau, EarlyStopping y ModelCheckpoint).
4. Evaluar con métricas, curvas de aprendizaje y matriz de confusión.


### División del dataset

Se separan los datos en tres subconjuntos estratificados (80/10/10) para poder monitorizar el rendimiento durante y después del entrenamiento.


In [ ]:
# División estratificada del dataset
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=33
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=33
)

print(f"Tamaño de entrenamiento: {X_train.shape}")
print(f"Tamaño de validación: {X_val.shape}")
print(f"Tamaño de prueba: {X_test.shape}")

for split_name, labels in [("Train", y_train), ("Validation", y_val), ("Test", y_test)]:
    counts = np.bincount(labels, minlength=len(CATEGORIES))
    resumen = ", ".join(f"{CATEGORIES[i]}: {count}" for i, count in enumerate(counts))
    print(f"{split_name} -> {resumen}")


### Data Augmentation y arquitectura de la CNN

El bloque convolucional sigue la filosofía usada en el notebook de CIFAR-10: bloques repetidos con dos convoluciones + *Batch Normalization*, *MaxPooling* y *Dropout* progresivo. Se añade un bloque de *data augmentation* al inicio para mejorar la generalización.


In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.08),
    layers.RandomZoom(0.1),
    layers.RandomContrast(0.1),
], name="data_augmentation")


def build_cnn_model(input_shape):
    inputs = keras.Input(shape=input_shape)
    x = data_augmentation(inputs)

    for filters, dropout_rate in zip([32, 64, 128], [0.15, 0.25, 0.35]):
        x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Conv2D(filters, (3, 3), padding="same", activation="relu")(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)

    x = layers.Flatten()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    return keras.Model(inputs, outputs, name="cnn_cats_dogs")


model = build_cnn_model((*IMG_SIZE, 3))
model.summary()


### Callbacks y entrenamiento

Se utilizan los mismos *callbacks* que en los apuntes: `ReduceLROnPlateau` para ajustar la tasa de aprendizaje, `EarlyStopping` para evitar sobreentrenamiento y `ModelCheckpoint` para conservar el mejor modelo.


In [ ]:
MODEL_PATH = "cnn_cats_dogs.keras"

callbacks = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ModelCheckpoint(filepath=MODEL_PATH, monitor='val_accuracy', save_best_only=True, verbose=1)
]

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=35,
    batch_size=32,
    callbacks=callbacks,
    verbose=1
)


### Evaluación cuantitativa y curvas de aprendizaje


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test, verbose=0)
print(f"Pérdida en test: {test_loss:.4f}")
print(f"Accuracy en test: {test_acc:.4f}")

history_df = pd.DataFrame(history.history)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history_df['loss'], label='Entrenamiento')
axes[0].plot(history_df['val_loss'], label='Validación')
axes[0].set_title('Pérdida')
axes[0].legend()
axes[1].plot(history_df['accuracy'], label='Entrenamiento')
axes[1].plot(history_df['val_accuracy'], label='Validación')
axes[1].set_title('Accuracy')
axes[1].legend()
plt.show()


### Matriz de confusión y ejemplos de predicciones


In [ ]:
y_pred_probs = model.predict(X_test, verbose=0).flatten()
y_pred = (y_pred_probs > 0.5).astype('int32')

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CATEGORIES, yticklabels=CATEGORIES)
plt.xlabel('Predicción')
plt.ylabel('Etiqueta real')
plt.title('Matriz de confusión')
plt.show()

print("
Classification Report:
")
print(classification_report(y_test, y_pred, target_names=CATEGORIES))

rng = np.random.default_rng(33)
sample_indices = rng.choice(len(X_test), size=min(6, len(X_test)), replace=False)
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for ax, idx in zip(axes.flatten(), sample_indices):
    ax.imshow(X_test[idx])
    ax.set_title(
        f"Real: {CATEGORIES[y_test[idx]]}
Pred: {CATEGORIES[y_pred[idx]]}
Perro: {y_pred_probs[idx]:.2f}"
    )
    ax.axis('off')
plt.tight_layout()
plt.show()


## Generación del CSV final de Predicciones

Una vez **definido y entrenado tu modelo** (no incluido en esta plantilla) y separadas tus imágenes de prueba en `X_test`, genera el archivo `predicciones_cats_dogs.csv` con el siguiente formato:

```text
id,label
0,0
1,0
2,1
...
```

Donde:
- `id` es el índice (comenzando en 0) de la imagen de prueba.
- `label` es `0` para gato y `1` para perro.

Notas importantes:
- Esta celda asume que tu **modelo** tiene una **salida sigmoide** (`Dense(1, activation='sigmoid')`) y que dispones de un `X_test` con las imágenes de evaluación (preprocesadas igual que X).
- Si usas una salida softmax de 2 neuronas, adapta la conversión de `y_pred` a clases.

In [ ]:
# Generación de CSV de predicciones
# Requisitos previos:
# - Debes haber definido y entrenado tu 'model'
# - Debes tener preparado 'X_test' con el mismo preprocesamiento (224x224, RGB, normalizado)

# Obtener predicciones del modelo
y_pred = model.predict(X_test, verbose=0)

# Si tu salida es sigmoide (1 neurona → binaria)
y_pred_classes = (y_pred > 0.5).astype("int32").flatten()

# Crear DataFrame con el formato solicitado
df = pd.DataFrame({
    "id": np.arange(1, len(y_pred_classes) + 1),
    "label": y_pred_classes
})

# Guardar como CSV
output_path = "predicciones_cats_dogs.csv"
df.to_csv(output_path, index=False)

print(f"Archivo guardado correctamente como: {output_path}")
df.head()